# Disaster-FCV Scan Tutorial

## Imports and Setup

In [1]:
import re
import os
import pprint
os.chdir("..")

from osgeo import gdal
from src.dfcv_colocation_mapping import data_download
from src.dfcv_colocation_mapping import common_utils
from src.dfcv_colocation_mapping import map_utils
from src.dfcv_colocation_mapping import widgets

import pandas as pd
from datetime import date
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

import warnings
warnings.filterwarnings(action="ignore", message=r"datetime.datetime.utcnow")
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

%load_ext autoreload
%autoreload 2

<a name="download-data"></a>
# Download Data

In [41]:
# Country name and admin Level
iso_code = "SLE" 
iso_code = re.sub(r'\([^)]*\)', '', iso_code).strip()
adm_level = "ADM3" 

# Conflict start and end dates
conflict_start_date = "2020-01-01" 
conflict_end_date = date.today()

# Displacement start and end years
displacement_start_year = 2020
displacement_end_year = 2025 

config_file = "src/dfcv_colocation_mapping/configs/data_config.yaml"
acled_cred_file = "creds/acled_creds.yaml"
dtm_cred_file = "creds/dtm_creds.yaml"
idmc_cred_file = "creds/idmc_creds.yaml"
adm_config_file = "src/dfcv_colocation_mapping/configs/adm_config.yaml"
osm_config_file = "src/dfcv_colocation_mapping/configs/osm_config.yaml"
acled_config_file = "src/dfcv_colocation_mapping/configs/acled_config.yaml"

dm = data_download.DatasetManager(
    iso_code, 
    adm_level=adm_level,
    config_file=config_file,
    acled_cred_file=acled_cred_file,
    dtm_cred_file=dtm_cred_file,
    idmc_cred_file=idmc_cred_file,
    adm_config_file=adm_config_file,
    osm_config_file=osm_config_file,
    acled_config_file=acled_config_file,
    conflict_start_date=conflict_start_date,
    displacement_start_year=displacement_start_year,
    displacement_end_year=displacement_end_year
)

## Optional: Filter Datasets

In [ ]:
data_category = "assets" # Choose from: "assets", "hazards", "conflict", "displacement", "osm"

def save_selection(result):
    dm.config[f"{data_category}_selected"] = result
    dm.set_selected_datasets()

selector = widgets.MultiSelector(
    data_category,
    dm.config[f"{data_category}_all"],
    dm.config[f"{data_category}_selected"],
    save_callback=save_selection
)
selector.show()

## Optional: Filter ACLED Categories

In [ ]:
drm_pillar = None #"risk_reduction"
asset_category = "demographic" 

dm.update_acled_selected(drm_pillar)
def save_selection(result):
    dm.acled_selected[asset_category] = result

selector = widgets.HierarchicalSelector(
    asset_category,
    hierarchy=dm.acled_hierarchy,
    selected=dm.acled_selected[asset_category],
    save_callback=save_selection,
    save_label="Save ACLED Selection"
)
selector.show()

## Download Datasets

In [42]:
%%time
dm.download_datasets()
geoplot = map_utils.GeoPlot(dm,  map_config_file="src/dfcv_colocation_mapping/configs/map_config.yaml")
dm.data.head(3)

INFO:root:Loading ADM3 geoboundaries...
INFO:root:Loading Asset Layers...
INFO:root:Loading Hazard Layers...
INFO:root:Loading ACLED data...
Processing worldpop: 100%|███████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.77it/s]
INFO:root:Loading IOM DTM data...
INFO:root:WARNING: Network connection to dtm.iom.int could not be established.
INFO:root:Loading IDMC GIDD data...
INFO:root:Loading UNHCR data...
INFO:root:Downloading OSM data...
100%|████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 18.86it/s]
INFO:root:Calculating multihazard scores...


CPU times: total: 1min 3s
Wall time: 1min 27s


,iso_code,ADM3,ADM3_ID,ADM2_ID,ADM2,ADM1_ID,ADM1,geometry,worldpop,earthquake,earthquake_worldpop_exposure_absolute,earthquake_worldpop_intensity_weighted_exposure_absolute,landslide_earthquake_gfdrr,landslide_earthquake_gfdrr_worldpop_exposure_absolute,landslide_earthquake_gfdrr_worldpop_intensity_weighted_exposure_absolute,landslide_precip_gfdrr,landslide_precip_gfdrr_worldpop_exposure_absolute,landslide_precip_gfdrr_worldpop_intensity_weighted_exposure_absolute,cyclone,cyclone_worldpop_exposure_absolute,cyclone_worldpop_intensity_weighted_exposure_absolute,drought_spei,drought_spei_worldpop_exposure_absolute,drought_spei_worldpop_intensity_weighted_exposure_absolute,heat_stress,heat_stress_worldpop_exposure_absolute,heat_stress_worldpop_intensity_weighted_exposure_absolute,wildfire_geos5,wildfire_geos5_worldpop_exposure_absolute,wildfire_geos5_worldpop_intensity_weighted_exposure_absolute,fluvial_flood_cdri,fluvial_flood_cdri_worldpop_exposure_absolute,fluvial_flood_cdri_worldpop_intensity_weighted_exposure_absolute,acled_worldpop_conflict_count,acled_worldpop_fatalities,acled_worldpop_fatalities_per_conflict,wbg_acled_worldpop_exposure_absolute,idmc_conflict_idp_total_2023,idmc_conflict_idp_total_2024,idmc_disaster_idp_total_2024,idmc_idp_total_2020,idmc_idp_total_2021,idmc_idp_total_2022,idmc_idp_total_2023,idmc_idp_total_2024,idmc_idp_total_2025,idmc_idp_mean,earthquake_worldpop_exposure_relative,earthquake_worldpop_intensity_weighted_exposure_relative,landslide_earthquake_gfdrr_worldpop_exposure_relative,landslide_earthquake_gfdrr_worldpop_intensity_weighted_exposure_relative,landslide_precip_gfdrr_worldpop_exposure_relative,landslide_precip_gfdrr_worldpop_intensity_weighted_exposure_relative,cyclone_worldpop_exposure_relative,cyclone_worldpop_intensity_weighted_exposure_relative,drought_spei_worldpop_exposure_relative,drought_spei_worldpop_intensity_weighted_exposure_relative,heat_stress_worldpop_exposure_relative,heat_stress_worldpop_intensity_weighted_exposure_relative,wildfire_geos5_worldpop_exposure_relative,wildfire_geos5_worldpop_intensity_weighted_exposure_relative,fluvial_flood_cdri_worldpop_exposure_relative,fluvial_flood_cdri_worldpop_intensity_weighted_exposure_relative,wbg_acled_worldpop_exposure_relative,mhs_hydrometerological_worldpop_exposure_absolute,mhs_hydrometerological_wbg_acled_worldpop_exposure_absolute,mhs_hydrological_worldpop_exposure_absolute,mhs_hydrological_wbg_acled_worldpop_exposure_absolute,mhs_meteorological_worldpop_exposure_absolute,mhs_meteorological_wbg_acled_worldpop_exposure_absolute,mhs_climatological_worldpop_exposure_absolute,mhs_climatological_wbg_acled_worldpop_exposure_absolute,mhs_all_worldpop_exposure_absolute,mhs_all_wbg_acled_worldpop_exposure_absolute,mhs_hydrometerological_worldpop_exposure_relative,mhs_hydrometerological_wbg_acled_worldpop_exposure_relative,mhs_hydrological_worldpop_exposure_relative,mhs_hydrological_wbg_acled_worldpop_exposure_relative,mhs_meteorological_worldpop_exposure_relative,mhs_meteorological_wbg_acled_worldpop_exposure_relative,mhs_climatological_worldpop_exposure_relative,mhs_climatological_wbg_acled_worldpop_exposure_relative,mhs_all_worldpop_exposure_relative,mhs_all_wbg_acled_worldpop_exposure_relative,mhs_hydrometerological_worldpop_intensity_weighted_exposure_relative,mhs_hydrological_worldpop_intensity_weighted_exposure_relative,mhs_meteorological_worldpop_intensity_weighted_exposure_relative,mhs_climatological_worldpop_intensity_weighted_exposure_relative,mhs_all_worldpop_intensity_weighted_exposure_relative
0,SLE,Dea,93885176B98792258194007,92492822B60042663074056,Kailahun,64854884B13787310286412,Eastern,"POLYGON ((-10.60592 7.86858, -10.606 7.9059, -...",17551.541016,0.031345,0.0,0.0,0.0,0.0,0.0,0.002465,3671.383057,137.205765,0,0.0,0.0,-1.041696,9882.917969,648.930298,2872.000000,0.0,0.0,13.308998,17551.541016,5049.248535,359.938517,1489.562500,243.646454,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2091

<a name="administrative-boundaries"></a>
# Administrative Boundaries

In [ ]:
group = "BLOC" 
adm_level = "ADM1" 

geoplot.plot_geoboundaries(
    adm_level=adm_level,
    group=group,
    save=True
);

<a name="asset-data-layers"></a>
# Asset Data Layers

In [7]:
widget = widgets.MapWidget(
    geoplot=geoplot,
    var_list=dm.asset_names,
    var_label="Asset",
    out_dir="assets"
)
widget.show()

Output()

In [ ]:
widget = widgets.MapWidget(
    geoplot=geoplot,
    plot_osm_networks=True,
    plot_osm_points=True,
    out_dir="osm"
)
widget.show()

<a name="internal-displacement-data-layers"></a>
# Displacement Data Layers

In [ ]:
widget = widgets.MapWidget(
    geoplot=geoplot,
    plot_displacement=True,
    out_dir="displacement"
)
widget.show()

In [12]:
widget = widgets.MapWidget(
    geoplot=geoplot,
    plot_displacement_points=True,
    out_dir="displacement"
)
widget.show()

Output()

<a name="conflict-data-layers"></a>
# Conflict Data Layers

In [34]:
widget = widgets.MapWidget(
    geoplot=geoplot,
    plot_conflict=True,
    plot_conflict_points=True,
    zoom_to_region=True,
    out_dir="conflicts"
)
widget.show()

Output()

## Plot Conflict Data Points Over Time

In [ ]:
geoplot.plot_timestamped(dm.acled["worldpop"], period="M")

<a name="conflict-exposure"></a>
# Conflict Exposure

In [50]:
widget = widgets.MapWidget(
    geoplot=geoplot,
    plot_conflict_exposure=True,
    out_dir="conflict_exposure"
)
widget.show()

Output()

<a name="hazard-data-layers"></a>
# Hazard Data Layers

In [ ]:
hazard = "earthquake" #"landslide_precip_gfdrr" # @param ["earthquake", "landslide_earthquake", "landslide_precip", "cyclone", "drought", "heat_stress", "fluvial_flood", "wildfire"]
ax = geoplot.plot_raster(hazard, save=True)
plt.show()

<a name="hazard-exposure"></a>
# Hazard Exposure

In [73]:
widget = widgets.MapWidget(
    geoplot=geoplot,
    #map_mode="bivariate_choropleth",
    plot_hazard_exposure=True,
    #plot_conflict_exposure=True,
    out_dir="hazard_exposure"
)
widget.show()

Output()

In [36]:
widget = widgets.MapWidget(
    geoplot=geoplot,
    plot_hazard_exposure=True,
    out_dir="hazard_exposure"
)
widget.show()

Output()

<a name="multihazard-exposure-score"></a>
# Multihazard Exposure Score

In [14]:
widget = widgets.MapWidget(
    geoplot=geoplot,
    plot_mhs_exposure=True,
    out_dir="multihazard_exposure"
)
widget.show()

Output()

<a name="multihazard-conflict-exposure-score"></a>
# Multihazard-Conflict Exposure Score

In [ ]:
widget = widgets.MapWidget(
    geoplot=geoplot,
    plot_conflict_exposure=True,
    plot_mhs_exposure=True,
    out_dir="multihazard_conflict_exposure"
)
widget.show()

## Bi-variate Choropleth Map

In [ ]:
widget = widgets.MapWidget(
    map_mode="bivariate_choropleth",
    geoplot=geoplot,
    zoom_to_region=True,
    plot_conflict_exposure=True,
    plot_mhs_exposure=True,
    out_dir="multihazard_conflict_exposure_bivariate"
)
widget.show()

<a name="interactive-mapping"></a>
# Interactive Mapping

In [ ]:
# @title Interactive Mapping with Folium
fmap = geoplot.plot_folium(
    adm_level=dm.adm_level,
    var="mhs_all_worldpop_exposure_relative",
    kwargs={"zoom_start": 7}
)
fmap = geoplot.plot_timestamped(
    dm.acled["worldpop"], fmap=fmap
)
fmap

In [ ]:
# @title Save Folium Map
fmap.save(f'{dm.iso_code}_folium_map.html')